# Complete LoRA & PEFT Implementation Guide - Notebook

### What is PEFT?
* PEFT (Parameter-Efficient Fine-Tuning) is a family of techniques designed to fine-tune large language models efficiently by updating only a small subset of parameters rather than the entire model.

### Key Concepts:

* Traditional Fine-tuning: Updates all model parameters (billions of parameters)
* PEFT: Updates only a small fraction (typically <1% of parameters)
* Memory Efficient: Requires significantly less GPU memory
* Storage Efficient: Only need to save the small adapter weights

### What is LoRA?
* LoRA (Low-Rank Adaptation) is a PEFT technique that decomposes weight updates into low-rank matrices, making fine-tuning extremely efficient.

### Use Cases:

* Domain Adaptation: Adapt general models to specific domains
* Task-Specific Fine-tuning: Create specialized versions for different tasks
* Personal Assistants: Create personalized model behavior
* Resource-Constrained Environments: When full fine-tuning isn't feasible

### Step 1: Installation & Dependencies

In [ ]:
!pip install transformers torch accelerate bitsandbytes peft
!pip install pypdf2 sentence-transformers faiss-cpu
!pip install datasets

### Step 2: Core Imports Library

In [ ]:
# Import PyTorch for tensor operations and deep learning
import torch

# Import required Hugging Face Transformers modules
from transformers import (
    AutoTokenizer,             # Tokenizer to preprocess text into tokens
    AutoModelForCausalLM,      # Pretrained causal language model (for text generation tasks)
    BitsAndBytesConfig,        # Config for quantization (e.g., 4-bit/8-bit model loading with QLoRA)
    TrainingArguments,         # Defines hyperparameters and settings for training
    pipeline                   # High-level utility to quickly run tasks like text generation, summarization, etc.
)

# Import PEFT (Parameter-Efficient Fine-Tuning) utilities
from peft import (
    LoraConfig,                # Configuration for LoRA (Low-Rank Adaptation) adapters
    get_peft_model,            # Function to wrap the base model with LoRA adapters
    TaskType,                  # Specifies the type of task (e.g., causal LM, seq2seq)
    prepare_model_for_kbit_training  # Prepares a model for low-bit (quantized) fine-tuning (QLoRA)
)

# Import PyPDF2 for extracting text content from PDF files
import PyPDF2

# Import SentenceTransformer for creating embeddings from text (useful for RAG pipelines)
from sentence_transformers import SentenceTransformer

# Import FAISS (Facebook AI Similarity Search) for fast vector search & similarity retrieval
import faiss

# Import NumPy for numerical operations (array, matrix computations)
import numpy as np

# Suppress warnings to keep output cleaner during training and execution
import warnings
warnings.filterwarnings('ignore')


### Step 3: PDF Processing Pipeline

In [ ]:
# Function to extract text from a PDF file
def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    text = ""  # Initialize an empty string to store extracted text

    # Open the PDF file in read-binary mode
    with open(pdf_path, 'rb') as file:
        # Create a PDF reader object
        pdf_reader = PyPDF2.PdfReader(file)

        # Loop through all pages in the PDF
        for page in pdf_reader.pages:
            # Extract text from each page and add a newline
            text += page.extract_text() + "\n"

    # Return the complete extracted text
    return text


# Function to split text into overlapping chunks
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks"""

    # Split the full text into words
    words = text.split()

    # Initialize a list to store text chunks
    chunks = []

    # Loop over the words in steps of (chunk_size - overlap) to create overlapping chunks
    for i in range(0, len(words), chunk_size - overlap):
        # Select a chunk of words from i to i + chunk_size
        chunk = ' '.join(words[i:i + chunk_size])

        # Append the chunk to the list
        chunks.append(chunk)

    # Return the list of overlapping chunks
    return chunks


### Step 4: Enhanced Vector Database

In [ ]:
# Define a simple vector database class using SentenceTransformers + FAISS
class SimpleVectorDB:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        # Load a pre-trained sentence transformer model for embeddings
        self.encoder = SentenceTransformer(model_name)

        # Initialize FAISS index (will be created after adding documents)
        self.index = None

        # Store original text chunks for retrieval
        self.chunks = []

    # Function to add text chunks into the vector database
    def add_documents(self, texts):
        """Add text chunks to vector database"""

        # Save the text chunks
        self.chunks = texts

        # Encode the text chunks into embeddings
        embeddings = self.encoder.encode(texts)

        # Get the dimension of the embedding vectors
        dimension = embeddings.shape[1]

        # Create a FAISS index for similarity search
        # IndexFlatIP = Inner Product similarity (works with normalized vectors → cosine similarity)
        self.index = faiss.IndexFlatIP(dimension)

        # Normalize embeddings to unit length for cosine similarity
        faiss.normalize_L2(embeddings)

        # Add embeddings to the FAISS index
        self.index.add(embeddings)

    # Function to search the most relevant text chunks given a query
    def search(self, query, k=3):
        """Search for most relevant chunks"""

        # Encode the query into an embedding
        query_embedding = self.encoder.encode([query])

        # Normalize the query embedding (for cosine similarity)
        faiss.normalize_L2(query_embedding)

        # Search top-k similar embeddings in the FAISS index
        scores, indices = self.index.search(query_embedding, k)

        # Collect results: retrieve text and similarity score
        results = []
        for i, idx in enumerate(indices[0]):
            results.append({
                'text': self.chunks[idx],  # Retrieve the original text chunk
                'score': scores[0][i]      # Store the similarity score
            })

        # Return the list of top-k results
        return results


### Step 5: LoRA Model Setup (Your Major Addition!)

In [ ]:
def setup_model_with_lora():
    """Setup model with LoRA configuration"""

    # Define quantization configuration for efficient memory usage
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                  # Enable 4-bit quantization for reduced memory usage
        bnb_4bit_quant_type="nf4",          # Use Normal Float 4 (NF4) quantization type
        bnb_4bit_compute_dtype=torch.float16, # Perform computations in 16-bit floating point precision
        bnb_4bit_use_double_quant=True      # Enable double quantization for further memory efficiency
    )

    # Choose a pre-trained model name
    model_name = "microsoft/DialoGPT-medium"  # Lightweight conversational model for quick experiments
    # Note: For higher accuracy, you can use "meta-llama/Llama-2-7b-chat-hf" (requires Hugging Face token)

    # Load tokenizer associated with the chosen model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Load the model with the quantization config and device map
    model = AutoModelForCausalLM.from_pretrained(
        model_name,                         # Model name to load
        quantization_config=bnb_config,     # Apply defined quantization settings
        device_map="auto",                  # Automatically place model layers on available devices (CPU/GPU)
        trust_remote_code=True              # Allow custom model code from Hugging Face repo
    )

    # Ensure tokenizer has a padding token (important for batching)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Use end-of-sequence token as padding if missing

    # Prepare model for training with k-bit quantization (optimizations for LoRA)
    model = prepare_model_for_kbit_training(model)

    # Define LoRA configuration
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,       # Task type: causal language modeling
        r=16,                               # Rank of LoRA decomposition
        lora_alpha=32,                      # Scaling factor for LoRA updates
        lora_dropout=0.1,                   # Dropout applied to LoRA layers during training
        target_modules=["c_attn", "c_proj"],# Specific layers in the model to apply LoRA
    )

    # Apply LoRA modifications to the model
    model = get_peft_model(model, lora_config)

    # Return both model and tokenizer for later use
    return model, tokenizer


### Step 6: Complete QA System Integration

In [ ]:
class PDFQuestionAnswering:
    def __init__(self):
        # Initialize the vector database for storing/retrieving document embeddings
        self.vector_db = SimpleVectorDB()
        # Placeholder for the language model
        self.model = None
        # Placeholder for the tokenizer
        self.tokenizer = None

    def load_pdf(self, pdf_path):
        """Load and process PDF"""
        print("Extracting text from PDF...")
        text = extract_text_from_pdf(pdf_path)  # Extract raw text from PDF

        print("Chunking text...")
        chunks = chunk_text(text)  # Split text into manageable overlapping chunks

        print("Creating vector database...")
        self.vector_db.add_documents(chunks)  # Add chunks to FAISS-based vector DB

        print(f"Processed {len(chunks)} chunks from PDF")  # Report number of chunks created

    def setup_model(self):
        """Initialize the model with LoRA"""
        print("Setting up model with LoRA...")
        self.model, self.tokenizer = setup_model_with_lora()  # Load LoRA-enhanced model & tokenizer
        print("Model setup complete!")  # Confirmation message

    def answer_question(self, question, max_length=200):
        """Answer question using retrieved context"""

        # Step 1: Retrieve most relevant text chunks from the vector DB
        relevant_chunks = self.vector_db.search(question, k=3)

        # Step 2: Combine retrieved chunks into a single context string
        context = "\n".join([chunk['text'] for chunk in relevant_chunks])

        # Step 3: Create a prompt that includes both context and user question
        prompt = f"""Based on the following context, answer the question:

Context: {context[:1000]}...  # Limit context to first 1000 chars for efficiency

Question: {question}

Answer:"""

        # Step 4: Tokenize the prompt for model input
        inputs = self.tokenizer.encode(
            prompt,                      # Full prompt (context + question)
            return_tensors="pt",         # Return PyTorch tensor
            truncation=True,             # Truncate if longer than max_length
            max_length=512               # Restrict input size to avoid memory issues
        )

        # Step 5: Generate answer from model without gradient computation (inference mode)
        with torch.no_grad():
            outputs = self.model.generate(
                inputs,                                   # Encoded input tokens
                max_length=inputs.shape[1] + max_length,  # Total output length (prompt + answer)
                num_return_sequences=1,                   # Generate one response
                temperature=0.7,                          # Sampling temperature (creativity level)
                pad_token_id=self.tokenizer.eos_token_id, # Use EOS as pad token
                do_sample=True                            # Enable sampling for varied responses
            )

        # Step 6: Decode model output back into text
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Step 7: Extract only the answer (remove prompt from the generated text)
        answer = response[len(prompt):].strip()

        # Step 8: Return structured response containing answer, context, and metadata
        return {
            'answer': answer,                                    # Final generated answer
            'context': context[:500] + "..." if len(context) > 500 else context,  # Truncated context
            'relevant_chunks': len(relevant_chunks)              # Number of chunks used for context
        }


### Step 7: Initialize the QA system

In [ ]:
def main():
    # Step 1: Initialize the PDF Question-Answering system
    qa_system = PDFQuestionAnswering()

    # Step 2: Provide instructions to upload the PDF file in Colab
    # Note: In Colab, you can use "from google.colab import files; files.upload()"
    # or mount Google Drive to access files.

    # Step 3: Define the path to your PDF file (replace this with actual path after upload)
    pdf_path = "Upload you File path"

    try:
        # Step 4: Load and process the PDF (extract text, chunk, and build vector DB)
        qa_system.load_pdf(pdf_path)

        # Step 5: Setup the model with LoRA configuration
        qa_system.setup_model()

        # Step 6: Define example questions related to the uploaded PDF
        questions = [
            "What payment security protocols are discussed?"  # Example for e-commerce/cybersecurity
        ]

        # Step 7: Loop through each question and generate answers
        for question in questions:
            print(f"\nQuestion: {question}")  # Print the current question
            result = qa_system.answer_question(question)  # Get answer using QA system

            # Step 8: Print the retrieved context (can also show 'answer' for better results)
            print(f"Answer: {result['context']}")

            # Step 9: Print a separator line for readability
            print("-" * 50)

    except FileNotFoundError:
        # Step 10: Handle error if file path is incorrect or file not uploaded
        print("Please upload your PDF file first!")
        print("Use: from google.colab import files; files.upload()")


### Step 8: Initialize question-answering session

In [ ]:
def interactive_qa():
    """Interactive question-answering session"""

    # Step 1: Initialize the PDF-based Question-Answering system
    qa_system = PDFQuestionAnswering()

    # Step 2: Import Colab file upload utility
    from google.colab import files

    # Step 3: Prompt user to upload a PDF file
    print("Please upload your PDF file:")
    uploaded = files.upload()  # Opens a file picker in Colab

    # Step 4: Extract the uploaded file name (path) from uploaded dictionary
    pdf_path = list(uploaded.keys())[0]

    # Step 5: Load and process the PDF (extract text, chunk, and store in vector DB)
    qa_system.load_pdf(pdf_path)

    # Step 6: Setup the LoRA-enhanced language model and tokenizer
    qa_system.setup_model()

    # Step 7: Notify user that the system is ready for interaction
    print("\nPDF loaded successfully! You can now ask questions.")
    print("Type 'quit' to exit.\n")

    # Step 8: Start an infinite loop for interactive Q&A
    while True:
        question = input("Your question: ")  # Take user question as input

        # Step 9: Exit condition if user types 'quit'
        if question.lower() == 'quit':
            break

        # Step 10: Get the answer from the QA system
        result = qa_system.answer_question(question)

        # Step 11: Print the model-generated answer
        print(f"\nAnswer: {result['answer']}\n")

        # Step 12: Print a separator line for better readability
        print("-" * 50)


### Step 9: Run and Test code

In [ ]:
if __name__ == "__main__":
    main()

Extracting text from PDF...
Chunking text...
Creating vector database...
Processed 61 chunks from PDF
Setting up model with LoRA...
Model setup complete!

Question: What payment security protocols are discussed?
Answer: Security is a critical concern for electronic payment systems (EPS) such as credit cards, debit cards, e - wallets, eChecks and smart cards, as they involve the transfer of sensitive financial information over the internet. To protect against fraud and unauthorized transactions, EPS providers employ a variety of security measures, including:  Encryption : EPS providers use encryption to protect sensitive information as it is transmitted over the internet. This makes it difficult for hackers to ...
--------------------------------------------------
